# 05 — Levels & evaluation (L0 -> L1 -> L2 -> L3)

The full stack on the 20 held-out forms:

- **L0** pretrained EasyOCR (from `predictions_l0.csv`)
- **L1** hybrid base read: digit comb cells -> **digit CNN**, insurance letters +
  free text -> **fine-tuned EasyOCR**, checkboxes -> nb-02 detection
- **L2** + vocab snap (`department`, `chief_complaint`, `allergies`,
  `medical_history`, `current_medications` -> their `data/vocab/` lists)
- **L3** + patient-DB match (SSN+insurance+name+DOB) -> identity fields replaced
  with the canonical DB values; the current visit stays from the form

Runs on the server (needs both checkpoints: `models/easyocr_ft/recognizer.pth`
and `models/digit_cnn/digit_cnn.pth`). Logic lives in `src/pipeline.py`,
`src/vocab_match.py`, `src/patient_match.py`.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import ocr, digit_model, vocab_match, patient_match, pipeline, finetune

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU = DEVICE == "cuda"
print("device:", DEVICE)

CROPS     = ROOT / "data" / "scans" / "crops"
FIELD_MAP = ocr.load_field_map(str(ROOT / "data" / "template" / "field_map.json"))
report    = ocr.load_checkbox_report(str(ROOT / "data" / "scans" / "preprocess_report.json"))
labels    = pd.read_csv(ROOT / "data" / "scans" / "labels.csv", dtype=str).fillna("")
labels_by_pid = {r["patient_id"]: dict(r) for _, r in labels.iterrows()}
label_cols = [c for c in labels.columns if c != "image"]
train_pids, val_pids = finetune.load_split(str(ROOT/"data"/"generated"/"split_train_val.json"))

EASYOCR_CKPT = ROOT / "models" / "easyocr_ft" / "recognizer.pth"   # text-only fine-tune
DIGIT_CKPT   = ROOT / "models" / "digit_cnn" / "digit_cnn.pth"

text_reader = ocr.get_finetuned_reader(str(EASYOCR_CKPT), gpu=GPU)
dmodel      = digit_model.load_model(str(DIGIT_CKPT), DEVICE)
db          = patient_match.load_db(str(ROOT / "data" / "generated" / "patient_db.json"))
vocab       = vocab_match.load_vocab(str(ROOT / "data" / "vocab"), db=db)   # db -> doctor vocab
print(f"loaded models + vocab + db ({len(db)} patients); evaluating {len(val_pids)} forms")

In [ ]:
# Run the full stack per form: hybrid L1 -> L2 -> L3.
rows = {"L1": [], "L2": [], "L3": []}
match_info = []; reviews = []
for pid in val_pids:
    res = pipeline.run_levels(str(CROPS/pid), FIELD_MAP, report.get(pid),
                              text_reader, dmodel, vocab, db, device=DEVICE)
    for lvl in ("L1","L2","L3"):
        rows[lvl].append({"patient_id": pid, **res[lvl]})
    match_info.append({"patient_id": pid,
                       **{k:v for k,v in res["match"].items() if k != "patient"}})
    reviews.append({"patient_id": pid, "flags": res["review"]})

def frame(rs):
    df = pd.DataFrame(rs)
    for c in label_cols:
        if c not in df.columns: df[c] = ""
    return df[label_cols].fillna("")
l1df, l2df, l3df = frame(rows["L1"]), frame(rows["L2"]), frame(rows["L3"])

# L0 from notebook 03/04 (filtered to the val forms)
l0df = pd.read_csv(ROOT/"data"/"generated"/"predictions_l0.csv", dtype=str).fillna("")
l0df = l0df[l0df.patient_id.isin(val_pids)]

GEN = ROOT/"data"/"generated"
l1df.to_csv(GEN/"predictions_l1_hybrid.csv", index=False)
l2df.to_csv(GEN/"predictions_l2.csv", index=False)
l3df.to_csv(GEN/"predictions_l3.csv", index=False)
print("wrote predictions_l1_hybrid / l2 / l3 .csv")

In [ ]:
# Patient-match accuracy (the L3 linchpin).
mi = pd.DataFrame(match_info)
correct = sum(r["patient_id"] == p for r, p in zip(match_info, val_pids))
print(f"top-1 patient match: {100*correct/len(val_pids):.1f}%  ({correct}/{len(val_pids)})")
print(f"matched above threshold: {mi['matched'].sum()}/{len(mi)}  "
      f"| avg score {mi['score'].mean():.2f}  avg margin {mi['margin'].mean():.2f}")

In [ ]:
# Per-field exact-match across the levels.
def norm(s): return " ".join(str(s if s is not None else "").strip().split()).lower()
gt = labels.set_index("patient_id")
def acc(df, f):
    d = df.set_index("patient_id")
    return 100*sum(norm(gt.loc[p,f])==norm(d.loc[p,f]) for p in val_pids if p in d.index)/len(val_pids)

fields = [c for c in label_cols if c != "patient_id"]
tab = pd.DataFrame({lvl: {f: acc(df,f) for f in fields}
                    for lvl,df in [("L0",l0df),("L1",l1df),("L2",l2df),("L3",l3df)]})
tab = tab[["L0","L1","L2","L3"]].round(1)
tab["best"] = tab.max(axis=1)
print(tab.sort_values("best", ascending=False).to_string())
print("\nMACRO:", {k: round(tab[k].mean(),1) for k in ["L0","L1","L2","L3"]})

## Final assembled record

Per the app routing: identity from L3 (db), current-visit + mutable fields from L2/form. This is what the app would surface for a matched patient.

In [ ]:
# Show the best-available value per field (final record) for a couple of forms.
for pid in val_pids[:2]:
    print(f"\n=== {pid} ===")
    row = l3df.set_index('patient_id').loc[pid]
    g = gt.loc[pid]
    for f in fields:
        mark = "OK " if norm(g[f])==norm(row[f]) else "xx "
        print(f"  {mark}{f:24s} {str(row[f])[:30]:30s}  (gt: {str(g[f])[:25]})")

## Review queue

Per the app design: DB-backed fields flag when the form *confidently disagrees* (possible update / bad match); form/visit fields flag when *invalid* (not a real date, out-of-vocab, bad format). Everything confident auto-accepts.

In [ ]:
# --- Review queue: fields the app flags for admin sign-off (not auto-accepted). ---
total = sum(len(r["flags"]) for r in reviews)
print(f"{total} flags across {len(reviews)} forms (invalid value, possible update, or no match)\n")
for r in reviews:
    if r["flags"]:
        print(r["patient_id"] + ":")
        for fld, reason in r["flags"].items():
            print(f"   [{fld}] {reason}")